### import dependancies

In [1]:
import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

/Users/hang/Documents/Development/ai-eng-course/ai-eng-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### download an example reference datapoint from langsmith

In [3]:
from dotenv import load_dotenv

load_dotenv("../../.env")

ls_client = Client()


In [4]:
dataset = ls_client.read_dataset(dataset_name="rag-evaluation-dataset")

In [6]:
list(ls_client.list_examples(dataset_id=dataset.id, limit =50))[15].outputs

{'ground_truth': 'You have a few options. The 3.5mm male plug to bare wire repair cables are useful for fixing headphones, microphones, or speakers. The iNassen splitter separates a 4-pole TRRS headset into separate audio and mic jacks. The Esinkin wireless audio receiver connects to speakers or receivers using RCA or 3.5mm input for wireless music streaming.',
 'reference_context_ids': ['B08DR4K78X', 'B07ZCLS3LL', 'B07FZ3G7SC'],
 'reference_descriptions': ['4 Pack Replacement 3.5mm Male Plug to Bare Wire Open End Headset TRRS Cord 4 Pole 1/8" Jack Stereo Audio Cable for Headphone Microphone Repair Cable 3.5mm Plug Connector for Earphone (12in) 4 Pack 12in Replacement 3.5mm male plug connector to bare wire audio cable Repair cable for broken headphone, microphone, speakers PIN 1 - red wire is the left channel; PIN 2 - white wire is the right channel; PIN 3 - green wire is mic; PIN 4 - black wire is ground Easy to install. Perfect for green hand. One year warranty. Worry-free money back

In [7]:
list(ls_client.list_examples(dataset_id=dataset.id, limit =50))[15].inputs

{'question': 'What do you have for audio connections or repairs with a 3.5mm jack?'}

In [ ]:
dataset # it is an uuid dataset

Dataset(name='rag-evaluation-dataset', description='RAG evaluation dataset', data_type=<DataType.kv: 'kv'>, id=UUID('2a0c144c-1259-4957-98d0-d6fb4c7a1fad'), created_at=datetime.datetime(2026, 7, 1, 2, 53, 52, 365776, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 7, 1, 2, 53, 52, 365776, tzinfo=TzInfo(0)), example_count=36, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-26.4-arm64-arm-64bit', 'sdk_version': '0.9.4', 'runtime_version': '3.12.13', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': None}})

In [16]:
reference_input = list(ls_client.list_examples(dataset_id=dataset.id, limit =50))[15].inputs
reference_output = list(ls_client.list_examples(dataset_id=dataset.id, limit =50))[15].outputs


now run on one example

### rag pipeline
pick up from production code, remvoe trace

In [11]:
import openai
from langsmith import traceable, get_current_run_tree 
# for cost calculation
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(url = "http://localhost:6333") # on docker image it called qdrant as defined

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata['usage_metadata'] = {
            "input_tokens": response.usage.prompt_tokens,
            "total_tokens": response.usage.total_tokens
        }
    return response.data[0].embedding

def retrieve_from_qdrant(query, k=5):
    response = qdrant_client.query_points(
        collection_name="amazon-electronics-items-collection-01",
        query=get_embedding(query),
        limit=k
    )
    retrieved_context_ids = []
    retrived_context = []
    similarity_scores = []
    retrived_context_ratings = []
    for point in response.points:
        retrieved_context_ids.append(point.payload['parent_asin'])
        retrived_context.append(point.payload['preprocessed_description'])
        similarity_scores.append(point.score)
        retrived_context_ratings.append(point.payload['average_rating'])
        
    return {"retrieved_context_ids": retrieved_context_ids,
            "retrived_context": retrived_context,
            "similarity_scores": similarity_scores,
            "retrived_context_ratings": retrived_context_ratings}

def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context['retrieved_context_ids'], context['retrived_context'], context['retrived_context_ratings']):
        formatted_context += f"- ID: {id}, Rating: {rating}, Description: {chunk}\n"
    return formatted_context


def build_prompt(preprocessed_context, question):
    prompt = f"""
    You are a helpful shopping assistant that can answer questions about the product in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the provided context only.
    - Never use word context and refer to it as the available product.
    - Do not use markdown formatting.

    Here is the context:
    {preprocessed_context}
    Here is the question:
    {question}
    """
    return prompt


def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt},
        ],
        reasoning_effort="none"
    )
    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata['usage_metadata'] = {
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens
        }
    return response.choices[0].message.content


def rag_pipeline(question, k=5):
    retrieved_context = retrieve_from_qdrant(question, k=k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)
    final_answer = {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieved_context['retrieved_context_ids'],
        "retrieved_context": retrieved_context['retrived_context']
    }
    return final_answer


In [12]:
rag_pipeline("can i get a charger?")

{'answer': 'Yes. We have a few options available:\n\n1) A USB to DC 4.0x1.7mm barrel jack power cable charger cord (2-pack, 6.6ft/2m) for devices like PSP and tablets/cellphones/laptops/netbooks.\n2) A KMC 6-outlet surge tap with 2 built-in USB charging ports (3.4A shared; up to 2.4A per USB port) for charging multiple devices.\n3) USB charging cables for specific devices (like Fitbit Blaze, or Ultimate Ears UE wireless charging dock cable-only, or USB-C to Lightning cables for iPhones—cable only).\n\nWhich device are you trying to charge (and what connector/brand)?',
 'question': 'can i get a charger?',
 'retrieved_context_ids': ['B076Q82C9F',
  'B0894DXZFP',
  'B08F9T8LLQ',
  'B09L19PLFM',
  'B01DD99H9Y'],
 'retrieved_context': ['Onite 2pcs of USB to DC 4.0x1.7mm Barrel Jack Power Cable Charger Cord for PSP 3000 2000 1000, Tablet, Cellphone, Laptop, Netbook, Electronics (6.6ft/2m) ',
  'ienza Long White 10Ft USB Power Charge Cable for Ultimate Ears UE Wireless Charging Dock, Boom 3, 

### RAGAS metrics

In [13]:
from ragas import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

/var/folders/wl/bfvmh_sd2_91_635wbryp9hm0000gn/T/ipykernel_35898/3504057058.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/wl/bfvmh_sd2_91_635wbryp9hm0000gn/T/ipykernel_35898/3504057058.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/wl/bfvmh_sd2_91_635wbryp9hm0000gn/T/ipykernel_35898/3504057058.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is depre

In [31]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


/var/folders/wl/bfvmh_sd2_91_635wbryp9hm0000gn/T/ipykernel_35898/2117773422.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
/var/folders/wl/bfvmh_sd2_91_635wbryp9hm0000gn/T/ipykernel_35898/2117773422.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [ ]:

sample = SingleTurnSample(
    question=reference_input['question'],
    answer=reference_output['answer'],
)

In [17]:
reference_input

{'question': 'What do you have for audio connections or repairs with a 3.5mm jack?'}

In [18]:
result = rag_pipeline(reference_input['question'])

In [ ]:
result # check if retrived ids are in reference data

{'answer': 'For audio connections/repairs with a 3.5mm jack, the available option is:\n\n1) B08DR4K78X (4 Pack)\nReplacement 3.5mm (1/8") male TRRS connector to bare-wire open-end headset cable (12in). It’s made for headphone/microphone repair. Pinout: red = left channel, white = right channel, green = mic, black = ground. Easy to install, one-year warranty, and worry-free money back.\n\nIf you meant a splitter instead of a repair cable, there’s also:\n2) B07ZCLS3LL\n3.5mm TRRS headset splitter adapter (3.5mm male to 2 dual female) for separating audio and mic jacks (compatible with Xbox One, PS4, phones, laptops).',
 'question': 'What do you have for audio connections or repairs with a 3.5mm jack?',
 'retrieved_context_ids': ['B08DR4K78X',
  'B07ZCLS3LL',
  'B07BHGM7MF',
  'B07MJPMQCH',
  'B008H7P8GY'],
 'retrieved_context': ['4 Pack Replacement 3.5mm Male Plug to Bare Wire Open End Headset TRRS Cord 4 Pole 1/8" Jack Stereo Audio Cable for Headphone Microphone Repair Cable 3.5mm Plug 

In [21]:
from tokenize import single_quoted

# Because scorer.single_turn_ascore(sample) is an async Ragas method.


async def ragas_context_precision_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)


In [22]:
await ragas_context_precision_id_based(result, reference_output)

0.4

In [23]:
async def ragas_context_recall_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [24]:
await ragas_context_recall_id_based(result, reference_output)

0.6666666666666666

In [32]:
async def ragas_faithfulness(run, example):

    sample = SingleTurnSample(
            user_input=run["question"],
            response=run["answer"],
            retrieved_contexts=run["retrieved_context"]
        )

    scorer = Faithfulness(llm=ragas_llm)
    
    return await scorer.single_turn_ascore(sample)

In [33]:
await ragas_faithfulness(result, reference_output)

0.5384615384615384

In [34]:
async def ragas_relevancy(run, example):

    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )

    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.single_turn_ascore(sample)

In [35]:
await ragas_relevancy(result, reference_output)

np.float64(0.831259473072521)